# 2. Rule-Based Phishing Detection Model

## 1. Foundation & Approach

### 1.1 Business Context

**Business Requirement:**
A small security consulting firm needs a way for their staff (security analysts and administrative personnel) to quickly determine whether a URL is phishing or legitimate. Staff currently evaluate suspicious URLs from emails manually, which is:
- Time-consuming (each URL takes 5-10 minutes to investigate)
- Inconsistent (depends on analyst experience)
- Not explainable to clients ("I just have a bad feeling about this URL")

**What they need:** A tool that says "This is phishing because X, Y, Z" with concrete reasons.

**What We Discovered - The Phishing Pattern:**
Our exploration of 235,795 URLs (42.8% phishing, 57.2% legitimate) from dataset4 revealed that **phishing sites are fundamentally simple**:

**The Phishing Formula (based on dataset4):**
1. **Zero resources** - 29.4% have no JavaScript, CSS, or images (69,212 URLs)
2. **No encryption** - 50.78% of phishing URLs use HTTP instead of HTTPS
3. **Minimal code** - 27.4% have zero complexity (64,486 URLs with bare HTML forms)
4. **No trust signals** - 13.3% lack basic indicators (31,407 URLs missing title, favicon, description, copyright)
5. **No professional standards** - 93.5% lack robots.txt, responsive design, or social media integration (71,930 URLs)

**Why this pattern exists:**
Phishing is **minimal effort credential harvesting**:
- Copy a login form (PayPal, Gmail, Bank)
- Host on cheap domain (.tk, .ml, .ga)
- POST form data to attacker's server
- Lives for hours/days, disappears before takedown

**Legitimate sites are the opposite:**
- Rich resources (JavaScript frameworks, CSS libraries, images)
- Professional standards (responsive, social media, SEO)
- Trust indicators (metadata, favicons, copyright)
- Code complexity (thousands of lines of JavaScript)

**Why Rule-Based Detection Works:**
These patterns are **not subtle** - they're obvious red flags a human would spot. A rule-based system can:
- Explain decisions clearly ("No HTTPS + zero resources + no trust signals = phishing")
- Be transparent (staff can verify rules make sense)
- Be maintainable (update rules as phishing tactics evolve)

**This Notebook:**
Build a proof-of-concept using these behavioral patterns, tested on historical data from dataset4. This is a **research prototype** to validate the approach, not a production-ready system.

### 1.2 Dataset Reality Check - Bias & Limitations

**The Circular Validation Problem:**

We are using dataset4 to both:
1. Discover the rules (e.g., "ResourceTypeScore = 0 → phishing")  
2. Test the rules (measure accuracy on the same dataset)

**What this means:**
- Accuracy on dataset4 shows the rules work on **this historical data**
- It does NOT prove the rules work on new, unseen phishing
- This is a proof-of-concept, not a production validation

**What we acknowledge:**
- No external validation dataset available
- Cannot claim generalization without out-of-sample testing
- Performance on real-world traffic is unknown

### 1.3 Why URL Patterns Alone Fail

**What We Found in Exploration:**

Some URL-based features are **strong indicators**:
- TLD = .top → 99.9% phishing (2,329 URLs)
- TLD = .edu → 99.7% legitimate (1,861 URLs)
- IsDomainIP = 1 → 100% phishing
- IsHTTPS = 0 → 100% phishing (50.78% of phishing caught)

These work because they're **structural absolutes** (educational institutions control .edu, IP addresses are suspicious).

**Where URL Patterns Fall Short:**

Many URL features are **correlations, not guarantees**:
- DomainLength > 30 → 92.2% phishing (but legitimate sites can have long domains)
- High digit count → strong signal (but api2024.example.io could be legitimate)
- Certain TLDs (.io, .co) → mixed (89.7% phishing for .io, but many legitimate startups use it)

URL-only detection misses the full picture.

**The Missing Piece: Webpage Behavior**

Even if a URL looks suspicious, the **webpage itself reveals the truth**:
- Does it have JavaScript, CSS, images? (29.4% of phishing have ZERO)
- Trust indicators present? (13.3% of phishing have NONE)
- Code complexity? (27.4% of phishing = bare HTML)
- Professional standards? (93.5% of phishing lack them)

**Why Both Matter:**

URL analysis catches obvious cases fast (no HTTPS, .top domain, IP address).  
Behavioral analysis catches sophisticated phishing (suspicious URL + simple webpage = phishing).

**Our Approach:**

Use URL features where they're strong, but **fetch and analyze the webpage** to get the complete picture.

### 1.4 The Real Signal - Behavioral Analysis

**What Behavioral Features Measure:**

Dataset4's 56 features can be grouped by what they evaluate:

**1. Resource Investment** (NoOfJS, NoOfCSS, NoOfImage)
- Measures: Developer effort, time investment
- Why it matters: Building a React app takes weeks; copying a login form takes minutes

**2. Trust Signals** (HasTitle, HasFavicon, HasDescription, HasCopyrightInfo)
- Measures: Attention to user experience, brand presence
- Why it matters: Legitimate businesses care about metadata; phishers copy-paste HTML

**3. Code Complexity** (LineOfCode, LargestLineLength)
- Measures: Functional depth vs static content
- Why it matters: Real sites have logic; phishing sites have forms that POST to external servers

**4. Professional Standards** (Robots, IsResponsive, HasSocialNet)
- Measures: Long-term business presence, SEO, user engagement
- Why it matters: Phishing sites live for hours/days; no point in responsive design or social media

**The Framework:**

These aren't random features - they're measuring **investment signals**:
- High investment (time, money, expertise) → likely legitimate
- Zero investment (bare minimum to steal credentials) → likely phishing

**Why This Generalizes (Hypothesis):**

Phishing economics don't change:
- Short lifespan → no ROI on professional development
- High volume, low success rate → minimal effort per site
- Disposable infrastructure → why build quality?

**What We're Testing:**

Can we detect phishing by measuring **effort invested in the webpage**?

### 1.5 Our Architecture - Live Website Analyzer

**The System Flow:**

```
URL input
    ↓
Fetch webpage (HTTP request, 5s timeout)
    ↓
Parse HTML content
    ↓
Extract relevant features
    ↓
Apply rules (covered in later sections)
    ↓
Output: verdict + confidence + explanation
```

**What Gets Extracted:**

**From the URL itself (fast, no fetch required):**
- Domain, TLD, length metrics
- Character counts (letters, digits, special chars)
- URL structure (subdomains, path, query params)

**From the webpage (requires fetch):**
- **Resource counts:** JavaScript files, CSS files, images
- **HTML metadata:** Title, favicon, description, copyright
- **Code metrics:** Lines of code, complexity measures
- **Professional indicators:** Robots.txt, responsive design, social media links
- **Security features:** HTTPS status
- **References:** Internal vs external links

**Why Fetching is Required:**

Dataset4 provides pre-extracted features. In production, we must:
1. Make HTTP request to the URL
2. Download HTML content
3. Parse and extract features needed for our rules
4. Then apply detection logic

**This is webpage analysis, not URL string matching.**

### 1.6 Scope of This Notebook

**What We're Building:**

**Phase 1: Feature Selection**
- Identify strongest URL features (IsHTTPS, IsDomainIP, TLD)
- Identify strongest behavioral features (ResourceTypeScore, TrustScore, ProfessionalScore)
- Justify selections based on exploration findings from dataset4

**Phase 2: Rule-Based Model**
- Build cascading rules using selected features
- Evaluate on dataset4 (train/test split)
- Measure: accuracy, precision, recall, false positive/negative rates
- Identify where rules succeed and where they fail

**Phase 3: ML Extension (Conditional)**
- **IF** rules show clear weaknesses (e.g., >5% false positive rate)
- **THEN** build ML model for edge cases
- Compare rule-based vs hybrid approach
- Maintain explainability requirement

**Decision Point:** ML is added **only if rules prove insufficient**. We test rules first.

**What's NOT in This Notebook:**
- Production data extraction tool (requires live URL fetching - dataset4 has pre-extracted features)
- Deployment architecture (API, scaling, monitoring)
- Continuous learning (feedback loops, model retraining)

**Success Criteria:**
- Clear feature selection rationale
- Rule-based model with documented performance
- Decision on whether ML is needed
- Explainable predictions for business requirement

## 2. Feature Selection

### 2.1 Data Loading & Preparation

**Our Approach: Full Dataset Validation**

We will evaluate our rules on **all 235,795 URLs** from dataset4. No train/test split.

**Why this makes sense:**

**1. We already explored everything**
- "1. Exploration.ipynb" analyzed all 235,795 URLs
- We discovered patterns (ResourceTypeScore=0 → phishing) from the full dataset
- Splitting now doesn't create "unseen" data - we already know what works

**2. Rule-based models are different from ML models**
- We're not training parameters that could overfit to specific samples
- We're applying known patterns discovered during exploration
- Rules are based on domain knowledge + dataset4 findings, not learned from a training algorithm

**3. Honest about limitations (from Section 1.2)**
- This validates: "Do our rules work on dataset4?"
- This does NOT validate: "Do our rules generalize to new phishing?"
- Circular validation is acknowledged, not hidden by splitting

**What we're loading:**
- Full dataset4 (235,795 URLs × 56 features)
- Exclude: URLSimilarityIndex (data leakage), FILENAME, URL, label (non-features)
- Remaining: 52 features for rule building

In [ ]:
import pandas as pd
import numpy as np

# Load dataset4
df = pd.read_csv('data/dataset4.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nPhishing (label=0): {(df['label'] == 0).sum()} ({(df['label'] == 0).sum() / len(df) * 100:.1f}%)")
print(f"Legitimate (label=1): {(df['label'] == 1).sum()} ({(df['label'] == 1).sum() / len(df) * 100:.1f}%)")

# Exclude problematic features
features_to_exclude = ['URLSimilarityIndex', 'FILENAME', 'URL', 'label']
feature_cols = [col for col in df.columns if col not in features_to_exclude]

print(f"\nFeatures available: {len(feature_cols)}")
print(f"Excluded: {features_to_exclude}")

In [ ]:
# Test some basic rules using raw features
# NOTE: label = 0 is Phishing, label = 1 is Legitimate

# Rule 1: NoOfJS + NoOfCSS + NoOfImage = 0 → phishing (zero resources)
zero_resources = (df['NoOfJS'] == 0) & (df['NoOfCSS'] == 0) & (df['NoOfImage'] == 0)
rule1_matches = df[zero_resources]
rule1_correct = (rule1_matches['label'] == 0).sum()  # Changed to 0 for phishing
print("Rule 1: Zero Resources (NoOfJS=0 AND NoOfCSS=0 AND NoOfImage=0) → Phishing")
print(f"  Matches: {len(rule1_matches)} URLs")
print(f"  Correct: {rule1_correct} ({rule1_correct / len(rule1_matches) * 100:.2f}% precision)")
print(f"  Coverage: {len(rule1_matches) / len(df) * 100:.1f}% of dataset")
print(f"  Phishing caught: {rule1_correct} out of {(df['label'] == 0).sum()} total phishing ({rule1_correct / (df['label'] == 0).sum() * 100:.1f}%)")
print()

# Rule 2: IsHTTPS = 0 → phishing (no encryption)
rule2_matches = df[df['IsHTTPS'] == 0]
rule2_correct = (rule2_matches['label'] == 0).sum()
print("Rule 2: IsHTTPS = 0 → Phishing")
print(f"  Matches: {len(rule2_matches)} URLs")
print(f"  Correct: {rule2_correct} ({rule2_correct / len(rule2_matches) * 100:.2f}% precision)")
print(f"  Coverage: {len(rule2_matches) / len(df) * 100:.1f}% of dataset")
print(f"  Phishing caught: {rule2_correct} out of {(df['label'] == 0).sum()} total phishing ({rule2_correct / (df['label'] == 0).sum() * 100:.1f}%)")
print()

# Rule 3: IsDomainIP = 1 → phishing
rule3_matches = df[df['IsDomainIP'] == 1]
rule3_correct = (rule3_matches['label'] == 0).sum()
print("Rule 3: IsDomainIP = 1 → Phishing")
print(f"  Matches: {len(rule3_matches)} URLs")
print(f"  Correct: {rule3_correct} ({rule3_correct / len(rule3_matches) * 100:.2f}% precision)")
print(f"  Coverage: {len(rule3_matches) / len(df) * 100:.1f}% of dataset")
print(f"  Phishing caught: {rule3_correct} out of {(df['label'] == 0).sum()} total phishing ({rule3_correct / (df['label'] == 0).sum() * 100:.1f}%)")
print()

# Rule 4: Zero trust signals (HasTitle=0 AND HasFavicon=0 AND HasDescription=0 AND HasCopyrightInfo=0)
zero_trust = (df['HasTitle'] == 0) & (df['HasFavicon'] == 0) & (df['HasDescription'] == 0) & (df['HasCopyrightInfo'] == 0)
rule4_matches = df[zero_trust]
rule4_correct = (rule4_matches['label'] == 0).sum()
print("Rule 4: Zero Trust Signals → Phishing")
print(f"  Matches: {len(rule4_matches)} URLs")
print(f"  Correct: {rule4_correct} ({rule4_correct / len(rule4_matches) * 100:.2f}% precision)")
print(f"  Coverage: {len(rule4_matches) / len(df) * 100:.1f}% of dataset")
print(f"  Phishing caught: {rule4_correct} out of {(df['label'] == 0).sum()} total phishing ({rule4_correct / (df['label'] == 0).sum() * 100:.1f}%)")

#### Summary: Four Strict Rules

**What the data shows:**

We tested four rules on dataset4's 100,945 phishing URLs:

1. **No HTTPS** (IsHTTPS = 0) → 100% precision, catches 50.8% of phishing
2. **Domain is IP address** (IsDomainIP = 1) → 100% precision, catches 0.6% of phishing
3. **Zero resources** (NoOfJS=0 AND NoOfCSS=0 AND NoOfImage=0) → 99.99% precision, catches 68.6% of phishing
4. **Zero trust signals** (HasTitle=0 AND HasFavicon=0 AND HasDescription=0 AND HasCopyrightInfo=0) → 99.97% precision, catches 31.1% of phishing

**The logical framework:**

These aren't statistical correlations we might overfit to. They're **red flags that make logical sense**:

- **No HTTPS?** No encryption = No security concern for user data
- **Domain is IP address?** Legitimate businesses use domain names, not 192.168.1.1
- **Zero resources?** No JavaScript, CSS, or images = Bare minimum effort
- **Zero trust signals?** No title, favicon, description, or copyright = No brand presence

**Why this reduces noise:**

These are **strict filters** that catch obvious phishing attempts. They work because:
- They align with how legitimate websites are built (encryption, branding, resources)
- They measure fundamental differences (investment vs disposable infrastructure)
- They're explainable to non-technical users ("This site has no encryption and no company information")

**Next steps:**

Investigate overlap between rules and determine total phishing coverage when combined.

### 2.2 Rule Overlap and Combined Coverage

Now we investigate:
1. How much overlap exists between rules (same URLs caught by multiple rules)
2. Total phishing coverage when combining all four rules
3. What phishing we're NOT catching (the remaining cases)

In [ ]:
# Calculate rule overlap and combined coverage

# Define the four rules
rule1 = (df['NoOfJS'] == 0) & (df['NoOfCSS'] == 0) & (df['NoOfImage'] == 0)  # Zero resources
rule2 = (df['IsHTTPS'] == 0)  # No HTTPS
rule3 = (df['IsDomainIP'] == 1)  # Domain is IP
rule4 = (df['HasTitle'] == 0) & (df['HasFavicon'] == 0) & (df['HasDescription'] == 0) & (df['HasCopyrightInfo'] == 0)  # Zero trust

# Get phishing URLs
phishing = df[df['label'] == 0]
total_phishing = len(phishing)

# Check which phishing URLs each rule catches - use loc to avoid reindexing warnings
caught_by_rule1 = phishing.loc[rule1[phishing.index]].index
caught_by_rule2 = phishing.loc[rule2[phishing.index]].index
caught_by_rule3 = phishing.loc[rule3[phishing.index]].index
caught_by_rule4 = phishing.loc[rule4[phishing.index]].index

# Combined: any URL caught by at least one rule
combined_mask = rule1 | rule2 | rule3 | rule4
caught_combined = phishing.loc[combined_mask[phishing.index]].index

print("=" * 70)
print("RULE OVERLAP ANALYSIS")
print("=" * 70)
print(f"\nTotal phishing URLs: {total_phishing:,}")
print()

# Individual coverage
print("Individual rule coverage:")
print(f"  Rule 1 (Zero resources):   {len(caught_by_rule1):,} ({len(caught_by_rule1)/total_phishing*100:.1f}%)")
print(f"  Rule 2 (No HTTPS):         {len(caught_by_rule2):,} ({len(caught_by_rule2)/total_phishing*100:.1f}%)")
print(f"  Rule 3 (Domain is IP):     {len(caught_by_rule3):,} ({len(caught_by_rule3)/total_phishing*100:.1f}%)")
print(f"  Rule 4 (Zero trust):       {len(caught_by_rule4):,} ({len(caught_by_rule4)/total_phishing*100:.1f}%)")
print()

# Combined coverage
print("Combined coverage (any rule matches):")
print(f"  Total caught: {len(caught_combined):,} ({len(caught_combined)/total_phishing*100:.1f}%)")
print(f"  Missed: {total_phishing - len(caught_combined):,} ({(total_phishing - len(caught_combined))/total_phishing*100:.1f}%)")
print()

# Overlap between rules
print("Overlap between rules:")
print(f"  Rule 1 AND Rule 2: {len(set(caught_by_rule1) & set(caught_by_rule2)):,}")
print(f"  Rule 1 AND Rule 4: {len(set(caught_by_rule1) & set(caught_by_rule4)):,}")
print(f"  Rule 2 AND Rule 4: {len(set(caught_by_rule2) & set(caught_by_rule4)):,}")
print()

# What's caught uniquely by each rule
print("Unique catches (caught ONLY by this rule):")
print(f"  Only Rule 1: {len(set(caught_by_rule1) - set(caught_by_rule2) - set(caught_by_rule3) - set(caught_by_rule4)):,}")
print(f"  Only Rule 2: {len(set(caught_by_rule2) - set(caught_by_rule1) - set(caught_by_rule3) - set(caught_by_rule4)):,}")
print(f"  Only Rule 3: {len(set(caught_by_rule3) - set(caught_by_rule1) - set(caught_by_rule2) - set(caught_by_rule4)):,}")
print(f"  Only Rule 4: {len(set(caught_by_rule4) - set(caught_by_rule1) - set(caught_by_rule2) - set(caught_by_rule3)):,}")

In [ ]:
# Analyze the phishing we're NOT catching

# Get missed phishing URLs
missed_phishing = phishing.loc[~combined_mask[phishing.index]]

print("=" * 70)
print("ANALYSIS OF MISSED PHISHING")
print("=" * 70)
print(f"\nMissed: {len(missed_phishing):,} phishing URLs ({len(missed_phishing)/total_phishing*100:.1f}%)")
print()

# What do missed phishing look like?
print("Characteristics of missed phishing:")
print(f"  Has HTTPS: {(missed_phishing['IsHTTPS'] == 1).sum():,} ({(missed_phishing['IsHTTPS'] == 1).sum()/len(missed_phishing)*100:.1f}%)")
print(f"  Has resources (JS/CSS/Images): {((missed_phishing['NoOfJS'] > 0) | (missed_phishing['NoOfCSS'] > 0) | (missed_phishing['NoOfImage'] > 0)).sum():,} ({((missed_phishing['NoOfJS'] > 0) | (missed_phishing['NoOfCSS'] > 0) | (missed_phishing['NoOfImage'] > 0)).sum()/len(missed_phishing)*100:.1f}%)")
print(f"  Has at least one trust signal: {((missed_phishing['HasTitle'] == 1) | (missed_phishing['HasFavicon'] == 1) | (missed_phishing['HasDescription'] == 1) | (missed_phishing['HasCopyrightInfo'] == 1)).sum():,} ({((missed_phishing['HasTitle'] == 1) | (missed_phishing['HasFavicon'] == 1) | (missed_phishing['HasDescription'] == 1) | (missed_phishing['HasCopyrightInfo'] == 1)).sum()/len(missed_phishing)*100:.1f}%)")
print()

print("Average resources in missed phishing:")
print(f"  Avg NoOfJS: {missed_phishing['NoOfJS'].mean():.1f}")
print(f"  Avg NoOfCSS: {missed_phishing['NoOfCSS'].mean():.1f}")
print(f"  Avg NoOfImage: {missed_phishing['NoOfImage'].mean():.1f}")
print()

# Sample of missed phishing URLs
print("Sample of missed phishing URLs:")
print(missed_phishing[['URL', 'IsHTTPS', 'NoOfJS', 'NoOfCSS', 'NoOfImage', 'HasTitle', 'HasFavicon']].head(10).to_string(index=False))

#### The Pareto Principle in Action

**What we observed:**

Four simple, logical rules catch **81% of phishing** with near-perfect precision. The remaining **19% are sophisticated** cases that require more complex analysis.

**This is the Pareto principle (80/20 rule):**
- 80% of results come from 20% of effort
- Simple rules handle the bulk of cases
- The last 20% requires 80% of the effort

**Why this matters:**

The pattern is universal across detection problems:
- Most phishing is lazy (no HTTPS, no resources, no trust signals)
- A small fraction invests effort (HTTPS, JavaScript, branding)
- Simple filters catch the majority, but edge cases need sophisticated methods

**For our business case:**

The security consulting firm can deploy these four rules immediately:
- Catch 81% of phishing with explainable decisions
- Flag the remaining 19% for manual review or ML-based analysis
- Staff time saved on the easy 81%, focused on the hard 19%

**The 19% we miss aren't failures - they're the actual security challenge.** Those sophisticated phishing sites (Firebase apps, IPFS hosting, etc.) would fool simpler detection anyway. The rules successfully separate noise from signal.

### 2.2 Feature Ranking Methodology

*[To be filled: How to measure 'strongest' features, composite features]*

### 2.3 Perfect Rules - 100% Precision Features

*[To be filled: Features with 100% precision from exploration]*

### 2.4 Strong Behavioral Features

*[To be filled: Features with >90% precision, investment indicators]*

### 2.5 Selected Features for Model

*[To be filled: Final feature list with justifications]*